# Rust 101: Syntax, Printing and Types

The first chapter of the Rust track. The goal is not to be exhaustive, but to get
the *shape* of the language right: how a program is organised, how values are
bound to names, which primitive types exist, and how to get things onto the screen
so you can see what you are doing.

Everything here is a **runnable example**. Read the markdown, run the cell below
it, and change nothing — the practice twin of this notebook is
`01_syntax-printing-and-types_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, not a beginner. The material is
pitched at someone arriving from **Python** and **C++**, so it spends its words on
where Rust *differs* rather than where it agrees. Ideas you already own — what a
function is, why floating point is approximate, what an integer overflow is — are
named once and moved past.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you
  reach it.** Most of the reasoning here is only convincing once you have seen the
  output.
- Cells are meant to be executed **in order**. Later cells deliberately lean on
  bindings established by earlier ones.
- Code that intentionally *fails* to compile is shown as a comment, together with
  the error you would get. Rust's diagnostics are teaching material in this track,
  not an obstacle to it.
- The prose is terse and the facts are dense. This is a reference you will come
  back to, not a novel you read once.

## The one idea to internalise

Rust is **strict**. The compiler rejects a great deal of code that Python or C++
would happily run: there are no implicit numeric conversions, no truthiness, no
reading of uninitialised values, no mutation of a binding you did not mark `mut`,
and no use of a value after it has been handed to someone else.

Almost every odd-looking rule in this chapter exists to make a category of bug
*impossible to express*. That is the trade: a slower first hour, in exchange for
entire classes of defect that simply cannot occur. Once a Rust program compiles,
much of the work that debugging used to do is already finished.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. Hello, Rust: The Shape of a Program](#1-hello-rust-the-shape-of-a-program)
- [2. Bindings and Mutability](#2-bindings-and-mutability)
- [3. Shadowing](#3-shadowing)
- [4. Primitive Scalar Types](#4-primitive-scalar-types)
- [5. Type Inference and Annotations](#5-type-inference-and-annotations)
- [6. Numeric Literals and Suffixes](#6-numeric-literals-and-suffixes)
- [7. Operators, Casting and Safe Arithmetic](#7-operators-casting-and-safe-arithmetic)
- [8. Floating-Point Notes](#8-floating-point-notes)
- [9. `char`, `&str` and `String`: A First Look](#9-char-str-and-string-a-first-look)
- [10. Printing and Formatting](#10-printing-and-formatting)
- [11. `format!`, `dbg!` and Debug vs Display](#11-format-dbg-and-debug-vs-display)
- [12. Tuples and Arrays](#12-tuples-and-arrays)
- [13. Constants, Statics and Type Aliases](#13-constants-statics-and-type-aliases)
- [14. Expressions, Statements and Blocks](#14-expressions-statements-and-blocks)
- [15. Reading the Compiler](#15-reading-the-compiler)
- [16. Cheat Sheet](#16-cheat-sheet)

The sections are cumulative: each assumes the ones above it. If you are returning
to look something up, the [cheat sheet](#16-cheat-sheet) at the end is the fastest
route back to a specific piece of syntax.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** Jupyter kernel. That is not a detail you can
ignore, because it changes the mental model of what a cell *is*.

Ordinarily you write a Rust program as a file containing items — chiefly
`fn main()` — and compile the whole thing. evcxr does something different: it
wraps whatever you type into a generated program, compiles it incrementally, and
keeps the results alive for the next cell. In effect you are typing the *body* of
`main`, with the compiler permanently running behind you.

### What that means in practice

- **No `fn main` in these cells.** The kernel supplies it. `main` is still how real
  programs start, and §1 shows its shape, but you will not write one here.
- **State persists across cells.** A binding introduced in cell 3 is still in scope
  in cell 12; a `fn`, `struct`, `const` or `use` defined anywhere is visible to
  every later cell. **Run the notebook top to bottom.**
- **Items are order-independent; statements are not.** Two function definitions in
  one cell may call each other freely, in either order. Two `let` statements may
  not use each other's names before they have actually run. This is the same
  distinction that makes Rust source files free to declare things in any order.
- **There is one namespace per notebook.** Two functions with the same name in
  different sections collide, exactly as they would in one big source file. When
  you write your own experiments, keep names distinct.
- **Re-running a cell that defines an item re-defines it.** If the kernel
  complains that something is already defined, *Restart* from the toolbar and run
  from the top. Restarting is cheap and always correct; treat it as the default
  fix, not a last resort.
- **A failing cell does not poison the kernel.** You get a compile error or a panic
  for that cell alone. Fix it and re-run just that cell.
- **`println!` writes to stdout.** `eprintln!` and `dbg!` write to stderr. In this
  notebook both land in the output area, occasionally with different colouring —
  which is a useful reminder that the two streams are genuinely separate (§10).

### The habit worth forming

When something behaves inexplicably after a series of edits, **restart the kernel
before investigating further**. Stale state is the single most common cause of
"impossible" behaviour in a notebook, and the diagnosis costs nothing.

The other habit, which matters much more: **read the compiler's message before
changing the code.** §15 dissects these messages properly, but the sooner you stop
skimming them, the faster this language becomes pleasant.


In [ ]:
// 0.1 Proof that state persists: this binding is visible in every later cell.
let kernel_marker = "state persists across cells";
println!("{kernel_marker}");


state persists across cells


## 1. Hello, Rust: The Shape of a Program

Rust is **ahead-of-time compiled** to native machine code. There is no interpreter,
no bytecode VM and no garbage collector at runtime. A build produces a standalone
binary that links against your platform's libc (or nothing at all); the compiler is
the only component that needs to be present.

### Items

A Rust source file is a sequence of **items** — top-level declarations. The kinds
you will meet in this track, roughly in order of first appearance:

| Item | Introduces |
|---|---|
| `fn` | a function |
| `const`, `static` | a compile-time constant, a global (§13) |
| `type` | an alias for an existing type (§13) |
| `use` | a path brought into scope |
| `struct`, `enum` | a product / sum type (chapter 10) |
| `trait`, `impl` | an interface and its implementations (chapter 10) |
| `mod` | a module — a namespace for other items |

Items may be declared in any order and may refer to each other freely. Rust resolves
names across the whole module before it compiles anything, which is why there is no
need for header files or forward declarations as C++ has.

### Where execution starts

The program begins at the item named `main`. Real programs look like this — shown as
a comment because the kernel already provides one:

``` rust
fn main() {
    println!("Hello, world!");
}
```

`main` takes no arguments and returns the **unit value** `()`, which is Rust's
"nothing meaningful" type. It may also return a `Result` to report failure to the
operating system, which is a pattern worth knowing: `fn main() -> Result<(), E>`
lets you use `?` in the body and exits non-zero on `Err`.

### Macros

`println!` is a **macro**, which is what the `!` signals. Macros are expanded into
ordinary code during compilation. In practice what you need to know is that macro
arguments are *not* evaluated like function arguments — the format string is parsed
by the macro itself, which is how `println!` can type-check your placeholders and
count your arguments at compile time. Other macros you will meet shortly:
`format!`, `vec!`, `assert_eq!`, `matches!`, `dbg!`.

### Statements, semicolons and comments

Ordinary statements end with `;`. §14 shows that the semicolon is not cosmetic
punctuation but the difference between an expression and a statement, which is the
single most important structural idea in the language. For now, follow the habit:
semicolons after statements, no semicolon on a block's final value.

Comments come in three forms:

- `//` to the end of the line.
- `/* ... */` for blocks — and unlike C, **these nest**, so you can comment out
  code that already contains a block comment.
- `///` is a **doc comment**: it attaches to the item that follows and is compiled
  into HTML documentation by `cargo doc`. `//!` does the same for the enclosing
  module.

### Naming and style

Formatting is not a matter of taste in Rust: `cargo fmt` is deterministic and the
ecosystem uses it everywhere. The conventions are enforced by a linter, `cargo
clippy`, and by long habit:

| Thing | Convention | Example |
|---|---|---|
| functions, variables, modules | `snake_case` | `parse_header` |
| types, traits, enum variants | `UpperCamelCase` | `HttpResponse` |
| constants, statics | `SCREAMING_SNAKE_CASE` | `MAX_RETRIES` |
| macros | `snake_case!` | `println!` |

Indentation is four spaces, and `use` declarations cluster at the top of a file.
Learning the conventions early costs nothing; fighting them costs a little every
day.


In [ ]:
// 1.1 The canonical program, shown as a comment because evcxr supplies it.

//     fn main() {
//         println!("Hello, world!");
//     }
//
// `fn main` takes no arguments and returns nothing (more precisely: it returns
// the unit value `()`). Cells in this notebook are statements only.

println!("Hello, Rust!");   // note the `!` — println is a macro
print!("no newline here ... ");
print!("... the line continues\n");

// Comments:
//   // a line comment
//   /* a block comment, which /* nests */ in Rust */
//   /// a doc comment — attaches to the item that follows (chapter 03)

// A leftover binding from §0 is still in scope, which proves the point:
println!("§0's marker is {kernel_marker:?}, {} bytes long", kernel_marker.len());


Hello, Rust!
no newline here ... ... the line continues
§0's marker is "state persists across cells", 27 bytes long


## 2. Bindings and Mutability

`let` introduces a **binding**: a name attached to a value. In Rust, bindings are
**immutable unless you say otherwise** — the opposite of Python's locals and of
C++'s `auto`.

``` rust
let x = 5;          // immutable
let mut x = 5;      // mutable
```

### Why immutable by default

This is not pedantry. A binding that cannot change is a name you can reason about
without tracking everything that happens between its declaration and its use. The
consequences show up later, in code that is heavily shared or run concurrently:
immutability is what makes Rust's guarantees about aliasing and data races
enforceable at all. Making it the default also means that *mutability is visible* —
seeing `mut` at a declaration tells you to look for the writes.

### What `mut` does and does not allow

- `mut` allows the **value** of the binding to change.
- `mut` never allows the **type** to change. `let mut x = 5; x = "five";` is an
  error, not a rebinding.
- `mut` is part of the binding, not the type. `&mut T` (chapter 05) is a different
  thing entirely and will be introduced carefully.

### Contrasts worth holding on to

| | Python | C++ | Rust |
|---|---|---|---|
| Default | mutable name | mutable variable | **immutable binding** |
| Opt in to mutation | — | — | `mut` |
| Type may change | yes | no | no (use shadowing, §3) |
| Must be initialised before use | no (runtime error) | mostly no | **yes, checked at compile time** |

### Two details that surprise people

**Assignment may copy or may move.** In Python, `let b = a` would be a second
reference to the same object. In Rust, `let b = a;` either *copies* the value (for
simple scalar types) or *transfers ownership* of it, leaving `a` unusable. The rule
depends on whether the type is `Copy`, and it is the subject of chapter 05.

**Deferred initialisation is legal.** You may declare a binding and assign it later,
and the compiler proves it is assigned on every path that reaches its use. That
means this compiles, and the failure mode it prevents — reading an uninitialised
variable — cannot happen:

``` rust
let answer: i32;
if condition { answer = 1; } else { answer = 2; }
println!("{answer}");
```

The compiler performs this "definite assignment" analysis for you, which is why a
Rust program has no undefined behaviour from uninitialised memory.

### Unused bindings

An unused binding is a warning, not an error, because it is almost always a mistake
worth flagging. The convention for "I meant to do this" is a leading underscore:

- `let _x = f();` — bind and drop, no warning.
- `let _ = f();` — discard the value entirely, no binding at all.
- `_` alone in a pattern means "match anything, ignore it".

Chapter 05 shows why `let _ = ...` and `let _x = ...` are not always equivalent for
owned values; for now, treat both as ways of saying "deliberately unused".


In [ ]:
// 2.1 Immutable by default, `mut` to opt in.

let x = 5;
println!("x = {x}");

// Reassigning an immutable binding does not compile:
//     x = 6;
//     error[E0384]: cannot assign twice to immutable variable `x`

let mut y = 5;
y += 1;                 // shorthand for  y = y + 1
y *= 2;
println!("y = {y}");

// `mut` changes the value, not the type.
let mut label = String::from("count=");
label.push_str("7");            // String is growable; see chapter 05
// label = 7;                   // error[E0308]: mismatched types
println!("{label}");

// Declared first, assigned later — legal, because there is exactly one path
// from the declaration to the use.
let later: i32;
later = 42;
println!("later = {later}");


x = 5
y = 12
count=7
later = 42


## 3. Shadowing

Writing `let` again with a name that already exists does not modify anything. It
introduces a **new binding** that hides the old one for the remainder of the scope.
The old value is not destroyed; it simply stops being reachable *by that name*.

| | `let mut x` | `let x = ...` again |
|---|---|---|
| Value can change | yes | yes (it is a fresh binding) |
| Type can change | **no** | **yes** |
| Old value still valid | yes | present but unreachable by name |
| Where it is idiomatic | in-place accumulation | step-by-step transformation |

### Shadowing versus mutation

Both let a name end up with a different value, so the choice looks arbitrary at
first. The distinction is about *intent*, and it shows up in the shape of the code:

- Use **`mut`** when you are accumulating in place — a running total, a counter, a
  buffer you are appending to. The value has one identity and it evolves.
- Use **shadowing** when you are transforming — each step produces a genuinely new
  value, often of a new type. The pipeline reads top to bottom and each line names
  its own result.

The payoff of shadowing is most visible when the type changes. Without it you would
need a naming scheme to keep the stages apart:

``` rust
let input_str = "  42  ";
let input_int = input_str.trim().parse::<i32>().unwrap();   // ugly
```

Shadowing removes the need for the suffixes entirely:

``` rust
let input = "  42  ";
let input = input.trim().parse::<i32>().unwrap();           // idiomatic
```

The reader sees one logical value passing through a series of transformations, and
the type errors that distinguish the stages are handled by the compiler.

### Scope rules

Shadowing is lexical. A `let` inside a block shadows only until that block's closing
brace, at which point the outer binding becomes visible again:

``` rust
let name = "outer";
{
    let name = "inner";
    println!("{name}");      // inner
}
println!("{name}");          // outer
```

Two consequences follow, and both are useful:

- A short-lived shadowing is a tidy way to reuse a name inside a helper block
  without leaking the change outwards.
- The right-hand side of a shadowing `let` sees the **previous** binding, so
  `let x = x + 1;` means what it appears to mean: read the old `x`, compute, then
  bind the result to a new `x`.

### Details that matter later

- **Shadowing is not mutation, and it does not free memory early.** For an owned
  value such as a `String`, the old value lives until the end of its scope unless
  the compiler can prove otherwise. When you reach chapter 05 and start caring about
  when resources are released, this distinction becomes practical rather than
  philosophical.
- **Function parameters can be shadowed inside the body** — `fn f(x: i32) { let x =
  x + 1; }` is legal and common. They simply cannot be *redeclared* in the
  signature.
- **Shadowing is not available in every position.** Pattern bindings in a `match`,
  for instance, introduce names by a slightly different mechanism (chapter 02).


In [ ]:
// 3.1 Shadowing, including a type change.

let value = 5;
let value = value * 2;
println!("value = {value}");

// Unlike `mut`, shadowing may change the type.
let tag = 42;                   // i32 by inference
let tag = tag.to_string();      // now a String
println!("tag = {tag:?}, bytes = {}", tag.len());

// Chained transformations therefore read naturally, left to right.
let raw = "  42  ";
let parsed: i32 = raw.trim().parse().unwrap();
let parsed = parsed + 1;
let parsed = format!("{parsed}!");
println!("parsed = {parsed}");

// Shadowing respects blocks: the outer binding reappears when the block ends.
let scope = "outer";
{
    let scope = "inner";
    println!("inside the block:  {scope}");
}
println!("outside the block: {scope}");

// Note that shadowing is not mutation: it introduces a new binding and merely
// makes the previous one unreachable by name.


value = 10
tag = "42", bytes = 2
parsed = 43!
inside the block:  inner
outside the block: outer


## 4. Primitive Scalar Types

Rust's scalar types are a fixed, closed set. The width of every integer is part of
its *name*, so there is no ambiguity to discover at runtime as there is with C's
`int`, and no arbitrary-precision integers as there are in Python.

| Kind | Types | Notes |
|---|---|---|
| Signed integers | `i8 i16 i32 i64 i128 isize` | `i32` is the default |
| Unsigned integers | `u8 u16 u32 u64 u128 usize` | `usize` for indices and lengths |
| Floats | `f32 f64` | `f64` is the default |
| Boolean | `bool` | exactly `true` / `false`, 1 byte |
| Character | `char` | one **Unicode scalar value**, 4 bytes |

### Choosing an integer

The type carries a promise about range, and the compiler checks it. Two rules of
thumb cover almost every case:

1. **Use `i32` or `i64` for ordinary arithmetic.** `i32` is the default and is fast
   on every platform; `i64` is a reasonable choice for anything that might
   accumulate. Signed arithmetic is easier to get right than unsigned, because
   differences and decrements cannot silently underflow.
2. **Use `usize` where the standard library does.** That means lengths, capacities
   and indices. `.len()` returns `usize`, indexing takes `usize`, and `Vec<T>` is
   parameterised over it. Fighting this — casting indices to `i32` and back — is
   pure friction.

Use `u8` for genuine byte data and the smaller unsigned widths when you are working
against a wire format or a file layout that specifies them. Otherwise, prefer
signed.

### `usize` and `isize` are pointer-sized

They are 64-bit on a 64-bit target and 32-bit on a 32-bit target. That is exactly
what you want for addressing memory, and exactly why they are a poor choice for
values that need a defined range in a file format or a network protocol.

### `char` is not a byte, and not a number

This is the biggest departure for anyone arriving from C or C++:

- A `char` is a **Unicode scalar value** and occupies **4 bytes**, always. `'a'`,
  `'ß'`, `'€'` and `'😀'` are each a single `char`.
- It is *not* a narrow integer. `'A' as u32 == 65` requires an explicit cast, and
  `'A' + 1` does not compile.
- A `char` is not a UTF-8 byte and cannot be used to index a string by position. §9
  explains why, and chapter 04 goes into it properly.

### No truthiness, no implicit conversions

`bool` is its own type and there is no coercion into or out of it. Every one of
these is an error, and correctly so:

``` rust
if 1 { }                // error: expected `bool`, found integer
let x = 1 + true;       // error: no `Add<bool>` for `{integer}`
let y: i64 = 1i32;      // error: mismatched types
```

In Python, truthiness makes `if items:` a convenient idiom and also a recurring
source of bugs when `items` is a number that happens to be `0`. Rust has no such
ambiguity: if you wrote a condition, it is a `bool`, and the reader can see what it
tests. §7 covers the conversion rules, such as they are.

### Sizes are a property of the target, not a promise

`size_of::<usize>()` is 8 on this machine and 4 on a 32-bit target. `size_of::<char>()`
is always 4, and `size_of::<bool>()` is always 1. When layout matters, measure it
with `std::mem::size_of` rather than assuming, and use fixed-width types when you
need a guarantee.


In [ ]:
// 4.1 One value of each kind, plus the sizes this machine actually uses.
use std::mem::size_of;

// Integers
let a: i8 = -128;                        // smallest i8
let b: u8 = 255;                         // largest u8
let c: i32 = -2_000_000_000;             // the default integer type
let d: u64 = 18_000_000_000_000_000_000;
let e: u128 = 18_000_000_000_000_000_000_000_000_000_000_000_000;
println!("{a} {b} {c} {d} {e}");

// usize / isize are pointer-sized; lengths and indices are usize.
let len: usize = 5;
println!("len = {len}, and usize is {} bytes wide", size_of::<usize>());

// Floats
let f: f32 = 1.5;
let g = 2.25;                            // f64 by default
println!("{f} {g}");

// bool and char
let ok: bool = true;
let letter: char = 'ß';                  // one code point, several UTF-8 bytes
println!("{ok} {letter}");

// Sizes on this machine
let types: [(&str, usize); 8] = [
    ("i8",   size_of::<i8>()),
    ("i32",  size_of::<i32>()),
    ("i64",  size_of::<i64>()),
    ("i128", size_of::<i128>()),
    ("f32",  size_of::<f32>()),
    ("f64",  size_of::<f64>()),
    ("char", size_of::<char>()),
    ("bool", size_of::<bool>())
];
for (name, bytes) in types {
    println!("{name:>5} = {bytes} byte(s)");
}
// println!("bool  = {} byte(s)", size_of::<bool>());

// No truthiness, and no implicit char/number conversion:
//     let bad = 1 + true;      // error[E0277]
//     let bad = 'A' + 1;       // error[E0369]


-128 255 -2000000000 18000000000000000000 18000000000000000000000000000000000000
len = 5, and usize is 8 bytes wide
1.5 2.25
true ß
   i8 = 1 byte(s)
  i32 = 4 byte(s)
  i64 = 8 byte(s)
 i128 = 16 byte(s)
  f32 = 4 byte(s)
  f64 = 8 byte(s)
 char = 4 byte(s)
 bool = 1 byte(s)


()

## 5. Type Inference and Annotations

Rust infers types, but it is a **local, flow-based** form of inference rather than
the whole-program inference of a language like Haskell. The compiler works out what
it can from the code it can see, and asks you when the answer is genuinely
ambiguous. In practice the rules are simple enough to predict.

### Where types come from

In rough order of precedence:

1. **An explicit annotation** — `let x: u64 = 10;`
2. **A literal suffix** — `let x = 10u64;`
3. **The context of use** — passing `x` to a function that takes `u64`, pushing it
   into a `Vec<u64>`, returning it where the signature says `u64`.
4. **The defaults** — an unconstrained integer literal is `i32`, an unconstrained
   float literal is `f64`.

Rule 4 is why `let n = 10;` needs no annotation. It is also the reason for the
`{integer}` and `{float}` placeholders you will see in error messages: until
something resolves them, the compiler genuinely does not know the width, and it says
so rather than guessing.

### What inference cannot do

**Inference never crosses a function boundary.** A function signature is a contract,
and its body is type-checked against the signature alone. The body cannot learn
anything from how the function is called, and the caller cannot learn anything from
how the body is written. This is a deliberate design choice: it means a function's
meaning can be understood, and its compilation checked, one function at a time.

The practical consequence is that you annotate parameters and return types, and you
almost never annotate locals:

``` rust
fn scale(n: i32, factor: i32) -> i32 {   // every part written out
    let product = n * factor;            // inferred: i32
    let label = format!("{product}");    // inferred: String
    let _ = &label;
    product
}
```

### When the compiler asks

The message is `type annotations needed` and it means the value could legitimately
be several types and nothing nearby decides which. The usual culprits are `parse`,
`collect`, `into` and `.sum()`. Fixes, in order of preference:

1. **Let the context decide.** Assign to a typed binding, or return where the
   signature fixes the type.
2. **Use a suffix** for a numeric literal: `0u8`, `1.0f32`.
3. **Use the turbofish**, `::<T>`, when you must name a generic argument inline:
   `"42".parse::<i32>()`, `Vec::<u8>::new()`, `iter.collect::<Vec<_>>()`.

The turbofish is worth getting used to now. It appears constantly from chapter 06
onwards, and the odd `::<>` syntax exists only because `<` is otherwise ambiguous
with the less-than operator.

### Annotations

The syntax is consistent and worth learning as a unit:

| Position | Syntax |
|---|---|
| binding | `let name: Type = value;` |
| function parameter | `fn f(name: Type)` |
| return type | `fn f() -> Type` |
| generic parameter | `fn f<T: Trait>(x: T)` |
| explicit type argument | `f::<T>(x)` |

There is no `auto`, no trailing return type after the name, and no separate
declaration/definition split. What you see in the signature is the whole contract.


In [ ]:
// 5.1 Where types come from.

// From the literal, with defaults applied.
let n = 10;          // i32
let r = 1.0;         // f64
let sum = n + 32;    // i32, because n is i32

// From an explicit annotation; the literal adapts to it.
let wide: u64 = 10;
let narrow: i8 = 10;

// From an inline suffix.
let via_suffix = 10u8;

// From a later use.
let ratio = 1.0 / 4.0;             // f64 (both operands are f64 literals)
let scaled: f32 = 1.0 / 4.0;       // f32, because the binding says so

println!("{n} {r} {sum} {wide} {narrow} {via_suffix} {ratio} {scaled}");


10 1 42 10 10 10 0.25 0.25


In [ ]:
// 5.2 Breaking ambiguity in `parse`.

// `parse` can produce any type that implements FromStr, so the target type must
// be decided by the context. Two equivalent ways to say it:
let by_annotation: i32 = "41".parse().unwrap();
let by_turbofish = "42".parse::<i32>().unwrap();
println!("{by_annotation} {by_turbofish}");

// Without either, the compiler stops:
//     let bad = "41".parse().unwrap();
//     error[E0282]: type annotations needed

// The turbofish is the general syntax for supplying generic parameters
// explicitly: `Vec::<u8>::new()`, `"42".parse::<i32>()`, `collect::<Vec<_>>()`.
// You will use it constantly from chapter 06 onwards.


## 6. Numeric Literals and Suffixes

Literal syntax is a thin lexical layer over the type checker, but it is dense with
convenience, and getting it right early avoids a class of confusing errors.

### Forms

| Form | Example | Notes |
|---|---|---|
| decimal | `1_000_000` | `_` is a visual separator and is ignored |
| hexadecimal | `0xFF`, `0xff` | case-insensitive digits |
| octal | `0o755` | |
| binary | `0b1010_1010` | |
| byte | `b'A'` | a `u8`, **not** a `char` |
| byte string | `b"hi"` | a `&[u8; 2]`, not a `&str` |
| char | `'a'`, `'\n'`, `'\\'`, `'\u{1F600}'` | a `char` |
| float | `1.5`, `0.25`, `1.5e-3` | exponent notation makes it a float |

The `_` separator is purely cosmetic to the compiler and enormously useful to a
human. `1_000_000` and `1000000` are the same `i32`; use the first whenever a
quantity has more than four digits, and group hexadecimal and binary literals on
nibble boundaries as a matter of habit.

### Suffixes

A suffix attaches to the whole literal and fixes its type:

``` rust
let a = 10u8;          // u8
let b = 1_000u16;      // u16
let c = 0.25f32;       // f32
let d = 10_usize;      // usize
```

A suffix and an annotation must agree. `let x: u8 = 10u16;` is an error, not a
conversion — nothing is implicitly narrowed or widened in Rust.

### Why the flavour of a literal matters

The literal is where the *first* constraint on a numeric type usually appears, and
therefore where inference typically resolves. A literal that does not fit its
inferred type is caught immediately:

``` rust
let x: u8 = 300;       // error: literal out of range for `u8`
let y: i8 = 128;       // error: literal out of range for `i8`
```

The same applies to negative unsigned literals, which are rejected at compile time
rather than wrapping silently as they do in C:

``` rust
let z: u32 = -1;       // error: cannot apply unary `-` to type `u32`
```

That message is a good illustration of the design philosophy: a construct that is
merely suspicious in C is an error here, and the error names the operation rather
than the symptom.

### Rendering in other bases

Literals parse in any base; printing in another base is a formatting concern rather
than a language one. The `{:b}`, `{:o}`, `{:x}` and `{:X}` specifiers covered in §10
convert on the way out. When you need to see the underlying bits of a value, that is
where to look.


In [ ]:
// 6.1 Every literal flavour.

let million = 1_000_000;              // underscores are ignored
let hex = 0xFF;
let octal = 0o755;
let binary = 0b1010_1010;
let byte = b'A';                      // u8, not char

let one_k: u16 = 1_000;               // annotation, no suffix
let tiny = 255u8;                     // suffix
let quarter = 0.25f32;

let milli = 1.5e-3;                   // 0.0015 as f64
let ten_billion = 2e10;

println!("{million} {hex} {octal} {binary} {byte} {one_k} {tiny} {quarter}");
println!("{milli} {ten_billion}");

// The same value, rendered in the other bases (more in §10).
println!("255 -> bin {:b}, oct {:o}, hex {:x}, HEX {:X}", 255, 255, 255, 255);

// An exponent literal is a float, so an integer binding rejects it:
//     let bad: i64 = 2e10;      // error[E0308]: mismatched types


1000000 255 493 170 65 1000 255 0.25
0.0015 20000000000
255 -> bin 11111111, oct 377, hex ff, HEX FF


## 7. Operators, Casting and Safe Arithmetic

### The operator inventory

| Category | Operators | Operand types |
|---|---|---|
| Unary | `-` `!` | signed numbers; `bool` (logical) and integers (bitwise) |
| Arithmetic | `+ - * / %` | numeric types |
| Comparison | `== != < <= > >=` | any type that implements the relevant traits |
| Logical | `&& \|\| !` | `bool` only — and `&&`/`\|\|` **short-circuit** |
| Bitwise | `& \| ^ << >>` | integer types |
| Compound assignment | `+= -= *= /= %= &= \|= ^= <<= >>=` | as above |

Arithmetic and bitwise operators are not built into the language; they are
implemented as **traits** (`Add`, `Sub`, `Mul`, `BitAnd`, and so on). That single
fact explains several things that look arbitrary at first — why `String + &str`
works while `&str + &str` does not, why you can define `+` for your own types
(chapter 10), and why mixing operand types is an error rather than a promotion.

### Precedence, and one place Rust fixes C

The ordering, strongest first:

```
unary - !  →  as  →  * / %  →  + -  →  << >>  →  &  →  ^  →  |  →  comparisons  →  &&  →  ||  →  ..
```

The bitwise operators sit **above** the comparisons, so `a & b == c` parses as
`(a & b) == c`. In C the opposite is true — `&` binds *looser* than `==` — which is
the source of a well-known class of C bug. Rust chose the less surprising order.
Comparison, `&&`, `||` and ranges bind loosest of the value operators, which is why
`if x < 10 && y > 0` reads the way it looks.

None of that is worth memorising. It is worth knowing that **parentheses cost
nothing and prevent everything**; use them whenever a reader might hesitate.

Note there is no ternary `?:`. In an expression position you use an `if` — see §14
and chapter 02.

### Integer division and remainder

- `/` on integers **truncates toward zero**. It does not floor.
- `%` takes the **sign of the left operand**, so `-5 % 3 == -2` and `5 % -3 == 2`.
  If you want a non-negative remainder, that is `rem_euclid`.
- **Division by zero always panics**, in every build profile. It is not undefined
  behaviour as in C, and it is not an `Option`.
- If floored division is what you need, `div_euclid` and `rem_euclid` say so
  explicitly.

### No implicit conversions

Rust performs **no** implicit numeric conversion, in either direction. Widening is
not automatic:

``` rust
let a = 1i32;
let b: i64 = a;        // error[E0308]: mismatched types
let c = a + 1i64;      // error[E0308]: mismatched types
```

This is the single most common early friction for anyone from C or Python, and it is
intentional: every conversion in a Rust program is visible at the point it happens.

### `as`: blunt, and exactly what it says

`as` is the built-in cast. It is deliberately *unchecked*, because its job is to
describe how the bits should be reinterpreted, not to validate anything:

- **Integer → integer** truncates to the target width and reinterprets, so
  `300i32 as u8 == 44` and `-1i32 as u8 == 255`.
- **Float → integer** truncates the fractional part... but only when the value is in
  range. Out-of-range values **saturate** (`1e300f64 as i32 == i32::MAX`) and `NaN`
  becomes `0`. It never wraps and never panics.
- **`bool` cannot be cast** to or from anything, and neither can `char` → `u8`; use
  `char as u32` and go from there.

When a lossless conversion exists, prefer `From`/`Into`, which is checked for you and
documents the intent. When it can fail, use `TryFrom`/`TryInto`, which returns a
`Result`:

| Intent | Reach for |
|---|---|
| bits should be reinterpreted | `as` |
| conversion cannot fail | `u64::from(x)` or `x.into()` |
| conversion may fail | `u8::try_from(x)?` |

### Overflow

For `+ - *` on integers the behaviour is **profile-dependent**: debug builds panic,
release builds wrap silently. This notebook runs in debug (you can confirm with
`cfg!(debug_assertions)`), so an unguarded overflow here will abort the cell with
`attempt to add with overflow`.

Silent wrapping in release is the dangerous case, and it is exactly the C behaviour
Rust would rather you did not rely on. When overflow is possible, say what you want:

| Method | Result on overflow |
|---|---|
| `checked_*` | `None` |
| `overflowing_*` | `(wrapped_value, true)` |
| `saturating_*` | clamps at `MIN` / `MAX` |
| `wrapping_*` | wraps, explicitly and on purpose |

These exist for `add`, `sub`, `mul`, `div`, `rem`, `pow`, `shl` and `shr`. Shifts have
their own rule: shifting by more than the width is an overflow error in debug, which
is why `1u8 << 8` panics.

### A first look at `Option` and `Result`

The `checked_*` methods return an **`Option<T>`**, which is an enum with exactly two
variants: `Some(value)` and `None`. There is no null in Rust; "possibly absent" is a
type, and the compiler makes you handle both cases. `Result<T, E>` is the same idea
for operations that can fail, with `Ok(value)` and `Err(error)`.

You have already met both, indirectly: `arr.get(i)`, `'5'.to_digit(10)` and
`"42".parse()` all return them. This chapter uses two escape hatches:

- `.unwrap()` — return the value, or **panic** if it is `None`/`Err`.
- `.expect("message")` — the same, with a message explaining what you expected.

Both are fine in a notebook, an example, or a place where a failure genuinely is a
bug. They are not fine in library code or in anything parsing untrusted input.
The honest ways to consume these types — `match`, `if let`, `?`, `unwrap_or` — are
chapter 02's business, and §8 and chapter 08 return to the error side of it.


In [ ]:
// 7.1 Operators: arithmetic, comparison, logical, bitwise, precedence.

// Arithmetic. `/` on integers truncates toward zero and `%` takes the sign of
// the LEFT operand.
let (a, b) = (7i32, 2i32);
println!("{a} + {b} = {}", a + b);
println!("{a} - {b} = {}", a - b);
println!("{a} * {b} = {}", a * b);
println!("{a} / {b} = {}   <- truncates, does not floor", a / b);
println!("-7 / 2 = {}, -7 % 3 = {}, 7 % -3 = {}",
         -7i32 / 2, -7i32 % 3, 7i32 % -3);
println!("-7.div_euclid(3) = {}, -7.rem_euclid(3) = {}",
         (-7i32).div_euclid(3), (-7i32).rem_euclid(3));

// `!` is bitwise on integers and logical on bool.
println!("!true = {}, !0u8 = {}, !5u8 = {}", !true, !0u8, !5u8);

// A comparison yields a bool, not an int. Convert explicitly when you need one.
let bigger = a > b;
println!("a > b = {bigger}, as a number: {}", u8::from(bigger));

// `&&` and `||` SHORT-CIRCUIT: the right operand is not evaluated when the left
// already decides the answer. This is what makes `if list.is_empty() || list[0] > 0`
// safe, and it is why guards are written in that order.
fn side_effect(label: &str) -> bool {
    println!("    (evaluated: {label})");
    true
}
let _ = false && side_effect("right of &&");
let _ = true || side_effect("right of ||");
let _ = true && side_effect("right of &&");

// Bitwise operators do NOT short-circuit; they act on every bit at once.
let (m, n) = (0b1100u8, 0b1010u8);
println!("{m:#010b} & {n:#010b} = {:#010b}", m & n);
println!("{m:#010b} | {n:#010b} = {:#010b}", m | n);
println!("{m:#010b} ^ {n:#010b} = {:#010b}", m ^ n);
println!("{m:#010b} << 2 = {:#010b}", m << 2);
println!("{m:#010b} >> 2 = {:#010b}", m >> 2);

// Precedence. In Rust the bitwise operators bind TIGHTER than the comparisons,
// so the first line below means `(m & n) == 8`. In C it would mean
// `m & (n == 8)`, which is a classic bug. Parenthesise anyway, for the reader.
println!("m & n == 8   -> {}", m & n == 8);
println!("(m & n) == 8 -> {}", (m & n) == 8);

// Compound assignment.
let mut total = 1u32;
total += 4;
total *= 10;
total -= 2;
total /= 4;
println!("total after arithmetic = {total}");
total <<= 4;
total |= 0b1;
println!("total after shifts and bit set = {total}");

// Division by zero always panics -- it is never undefined behaviour and never
// silently produces garbage:
//     let boom = 1u32 / 0;
//     panicked: attempt to divide by zero
println!("1 / 1 = {}", 1u32 / 1);


7 + 2 = 9
7 - 2 = 5
7 * 2 = 14
7 / 2 = 3   <- truncates, does not floor
-7 / 2 = -3, -7 % 3 = -1, 7 % -3 = 1
-7.div_euclid(3) = -3, -7.rem_euclid(3) = 2
!true = false, !0u8 = 255, !5u8 = 250
a > b = true, as a number: 1
    (evaluated: right of &&)
0b00001100 & 0b00001010 = 0b00001000
0b00001100 | 0b00001010 = 0b00001110
0b00001100 ^ 0b00001010 = 0b00000110
0b00001100 << 2 = 0b00110000
0b00001100 >> 2 = 0b00000011
m & n == 8   -> true
(m & n) == 8 -> true
total after arithmetic = 12
total after shifts and bit set = 193
1 / 1 = 1


In [ ]:
// 7.2 `as` reinterprets bits; From/Into and TryFrom/TryInto do not.

// Integer -> integer truncates to the target width, then reinterprets.
let big: i32 = 300;
println!("300i32 as u8 = {}   <- 300 % 256", big as u8);
println!("-1i32 as u8  = {}   <- two's complement wrap", -1i32 as u8);
println!("300i32 as i8 = {}", big as i8);

// Float -> integer truncates the fraction when the value is IN RANGE...
println!("2.9f64 as i32  = {}", 2.9f64 as i32);
println!("-2.9f64 as i32 = {}", -2.9f64 as i32);
// ...and SATURATES when it is not. It never wraps, and it never panics.
println!("1e300f64 as i32  = {}   <- i32::MAX", 1e300f64 as i32);
println!("-1e300f64 as i32 = {}   <- i32::MIN", -1e300f64 as i32);
println!("f64::NAN as i32  = {}   <- NaN becomes 0", f64::NAN as i32);

// `char` converts with `as` too, but only in the direction char -> integer.
println!("'A' as u32 = {}, '€' as u32 = {}", 'A' as u32, '€' as u32);

// Widening, three ways. Prefer the last two: they are checked, and they state
// the intent rather than the bit layout.
let from_as: i64 = big as i64;
let from_from: i64 = i64::from(42i32);
let from_into: i64 = 42i32.into();
println!("{from_as} {from_from} {from_into}");

// When a conversion can FAIL, `as` is the wrong tool -- it would silently
// produce a wrong number. TryFrom returns a Result instead.
println!("u8::try_from(200i32) = {:?}", u8::try_from(200i32));
println!("u8::try_from(300i32) = {:?}", u8::try_from(300i32));
println!("u8::try_from(300i32).unwrap_or(0) = {}",
         u8::try_from(300i32).unwrap_or(0));

// There are NO implicit numeric conversions, in either direction:
//     let bad = 1i32 + 1i64;    // error[E0308]: mismatched types
//     let bad: i64 = 1i32;      // error[E0308]: mismatched types

// Division and remainder, for the record.
println!("7 / 2 = {}, -7 / 2 = {}", 7 / 2, -7 / 2);
println!("7 % 2 = {}, -7 % 2 = {}", 7 % 2, -7 % 2);


300i32 as u8 = 44   <- 300 % 256
-1i32 as u8  = 255   <- two's complement wrap
300i32 as i8 = 44
2.9f64 as i32  = 2
-2.9f64 as i32 = -2
1e300f64 as i32  = 2147483647   <- i32::MAX
-1e300f64 as i32 = -2147483648   <- i32::MIN
f64::NAN as i32  = 0   <- NaN becomes 0
'A' as u32 = 65, '€' as u32 = 8364
300 42 42
u8::try_from(200i32) = Ok(200)
u8::try_from(300i32) = Err(TryFromIntError(PosOverflow))
u8::try_from(300i32).unwrap_or(0) = 0
7 / 2 = 3, -7 / 2 = -3
7 % 2 = 1, -7 % 2 = -1


In [ ]:
// 7.3 The four overflow-aware arithmetic families.

// In a debug build (this notebook's default) an unguarded overflow PANICS:
//     let boom = 200u8 + 100u8;
//     panicked: attempt to add with overflow
// In a release build it would wrap silently instead. Neither is behaviour you
// asked for, so when overflow is possible, say which one you want.
println!("cfg!(debug_assertions) = {}  <- this notebook is a debug build",
         cfg!(debug_assertions));

let m: u8 = 250;

println!("checked_add(5)       -> {:?}", m.checked_add(5));       // Some(255)
println!("checked_add(10)      -> {:?}", m.checked_add(10));      // None
println!("wrapping_add(10)     -> {}", m.wrapping_add(10));       // 4
println!("saturating_add(10)   -> {}", m.saturating_add(10));     // 255

let (value, did_overflow) = m.overflowing_add(10);
println!("overflowing_add(10)  -> {value}, overflowed = {did_overflow}");

// The same families exist for sub, mul, div, rem, pow, shl and shr.
println!("u8::MAX              -> {}", u8::MAX);
println!("i8::MIN              -> {}", i8::MIN);
println!("100u8.checked_mul(3) -> {:?}", 100u8.checked_mul(3));   // None
println!("100u8.wrapping_mul(3)-> {}", 100u8.wrapping_mul(3));    // 44
println!("2u8.checked_pow(10)  -> {:?}", 2u8.checked_pow(10));    // None
println!("2u8.saturating_pow(10)-> {}", 2u8.saturating_pow(10));  // 255
println!("1u8.checked_sub(2)   -> {:?}", 1u8.checked_sub(2));     // None

// Shifts follow the same rule: shifting by at least the width is an overflow.
//     let boom = 1u8 << 8;
//     panicked: attempt to shift left with overflow
println!("1u8 << 7             -> {:#010b}", 1u8 << 7);


In [ ]:
// 7.4 `Option`, `Result`, and the two escape hatches.

// `checked_*` returns Option<u8>: either Some(value) or None. There is no null
// in Rust and no sentinel value -- absence is part of the type.
let room_left: u8 = 250;
println!("250u8.checked_add(5)  = {:?}", room_left.checked_add(5));
println!("250u8.checked_add(10) = {:?}", room_left.checked_add(10));

// You cannot use an Option as though it were the number. That is the whole
// point, and it is why `Option` prevents the class of bug that null causes:
//     let sum = room_left.checked_add(10) + 1;
//     error[E0369]: cannot add `{integer}` to `Option<u8>`

// Escape hatch 1: unwrap. Gives the value, or panics on None.
println!("checked_add(5).unwrap() = {}", room_left.checked_add(5).unwrap());
//     room_left.checked_add(10).unwrap();
//     panicked: called `Option::unwrap()` on a `None` value

// Escape hatch 2: expect, which panics with your own explanation instead.
println!("checked_add(5).expect(..) = {}",
         room_left.checked_add(5).expect("250 + 5 must fit in a u8"));

// The non-panicking readers, for when absence is an ordinary outcome.
println!("unwrap_or(0)        = {}", room_left.checked_add(10).unwrap_or(0));
println!("unwrap_or_default() = {}", room_left.checked_add(10).unwrap_or_default());
println!("is_some() / is_none() = {} / {}",
         room_left.checked_add(5).is_some(),
         room_left.checked_add(10).is_none());
// `map` transforms the value inside Some and leaves None alone.
println!("20u8.checked_add(5).map(x2) = {:?}",
         20u8.checked_add(5).map(|v| v * 2));
println!("250u8.checked_add(10).map(x2) = {:?}",
         room_left.checked_add(10).map(|v| v * 2));

// `Result` has the same shape, but carries an error instead of nothing.
let good: Result<i32, _> = "42".parse();
let bad: Result<i32, _> = "nope".parse();
println!("good = {good:?}");
println!("bad  = {bad:?}");
println!("bad.unwrap_err()   = {}", bad.clone().unwrap_err());
println!("bad.unwrap_or(-1)  = {}", bad.as_ref().unwrap_or(&-1));
println!("good.unwrap_or(-1) = {}", good.as_ref().unwrap_or(&-1));

// Both types are ordinary enums, and `match` is the honest way to take them
// apart. That is chapter 02's subject; this is just a preview.
let parsed: Result<i32, _> = "42".parse();
let number = match parsed {
    Ok(n) => n,
    Err(_) => -1,
};
println!("match gave {number}");


## 8. Floating-Point Notes

`f32` occupies 32 bits and carries roughly 7 significant decimal digits; `f64`
occupies 64 bits and carries 15–17. Unsuffixed literals **with a decimal point**
default to `f64`. Both implement **IEEE-754** binary floating point, so they behave
like C's `double` and like Python's `float` — Python's `float` *is* an `f64`.

Four consequences follow, and they are not edge cases:

1. **Most decimal fractions have no exact binary representation.** `0.1` is a
   repeating fraction in base 2, exactly as `1/3` is in base 10. So `0.1 + 0.2`
   lands one ulp away from `0.3`, and `==` reports `false`.
2. **Division by zero does not panic.** It yields `±inf`, and `0.0 / 0.0` yields
   `NaN`. This is the one place where Rust deliberately does *not* stop you —
   because IEEE-754 defines the answer.
3. **`NaN != NaN`.** NaN poisons every comparison: `==`, `<`, `>` and `<=` are all
   false against it. Test with `.is_nan()`, never with equality.
4. **Therefore floats are not `Ord` and not `Eq`.** They implement only
   `PartialEq` and `PartialOrd`. That has real knock-on effects: `[f64].sort()`
   does not exist (bisection needs a total order), and `HashMap<f64, _>` does not
   exist either. `.sort_by(f64::total_cmp)` supplies the total order instead.

`f64::EPSILON` is the gap between `1.0` and the next representable value — a useful
constant, but **not** a universal tolerance. Away from 1.0 the gap grows, so choose
your tolerance from the magnitude of the data or from how many significant digits
you actually trust.

Practical rules:

- Never compare floats for equality. Compare an absolute difference against a
  tolerance, or use `total_cmp` when you need a sort.
- Default to `f64`. Reach for `f32` only for memory-bound bulk data, having
  measured.
- Do not store money in floats. Store integer minor units (cents) or a decimal
  type.
- `as` truncates and saturates (see §7.2), so call `.round()` first if that is what
  you meant.
- Floats and integers never mix implicitly: `1 + 1.0` is `error[E0277]`.


In [ ]:
// 8.1 Representation, and why `==` is wrong.

let a: f32 = 0.1 + 0.2;
let b: f64 = 0.1 + 0.2;
println!("f32 0.1 + 0.2 = {a:.20}");
println!("f64 0.1 + 0.2 = {b:.20}");
println!("b == 0.3          -> {}", b == 0.3);

// Compare with a tolerance instead.
let tolerance = 1e-12;
println!("(b - 0.3).abs() < 1e-12  -> {}", (b - 0.3).abs() < tolerance);


f32 0.1 + 0.2 = 0.30000001192092895508
f64 0.1 + 0.2 = 0.30000000000000004441
b == 0.3          -> false
(b - 0.3).abs() < 1e-12  -> true


In [ ]:
// 8.2 Infinities, NaN, and the missing Ord.

let nan = 0.0f64 / 0.0;
let inf = 1.0f64 / 0.0;
println!("nan = {nan}, inf = {inf}");
println!("nan == nan         -> {}", nan == nan);        // false
println!("nan != nan         -> {}", nan != nan);        // true
println!("nan.is_nan()       -> {}", nan.is_nan());
println!("inf.is_infinite()  -> {}", inf.is_infinite());
println!("inf > f64::MAX     -> {}", inf > f64::MAX);

// f64 is not Ord, so `.sort()` is unavailable. `total_cmp` supplies a total
// order that is well-defined even in the presence of NaN.
let mut values = [3.0f64, f64::NAN, 1.0, -2.5];
values.sort_by(f64::total_cmp);
println!("sorted with total_cmp: {values:?}");


nan = NaN, inf = inf
nan == nan         -> false
nan != nan         -> true
nan.is_nan()       -> true
inf.is_infinite()  -> true
inf > f64::MAX     -> true
sorted with total_cmp: [-2.5, 1.0, 3.0, NaN]


In [ ]:
// 8.3 Rounding, powers, roots, logs.

println!("floor(2.7) = {}, ceil(2.1) = {}, round(2.5) = {}",
         2.7f64.floor(), 2.1f64.ceil(), 2.5f64.round());
println!("trunc(-2.7) = {}, fract(2.75) = {}",
         (-2.7f64).trunc(), 2.75f64.fract());
println!("abs(-3.5) = {}, signum(-3.5) = {}",
         (-3.5f64).abs(), (-3.5f64).signum());
println!("sqrt(2) = {:.6}", 2.0f64.sqrt());
println!("2^10 = {}, 2^0.5 = {:.6}", 2.0f64.powi(10), 2.0f64.powf(0.5));
println!("e^1 = {:.6}, ln(10) = {:.6}, log10(1000) = {}",
         1.0f64.exp(), 10.0f64.ln(), 1000.0f64.log10());
println!("PI = {}, EPSILON = {:e}", std::f64::consts::PI, f64::EPSILON);

// Integers and floats never mix implicitly:
//     let bad = 1 + 1.0;   // error[E0277]: cannot add `{float}` to `{integer}`

// A common trap: `as` on a float truncates toward zero, it does not round.
println!("2.9 as i32         -> {}", 2.9f64 as i32);
println!("2.9.round() as i32 -> {}", 2.9f64.round() as i32);


floor(2.7) = 2, ceil(2.1) = 3, round(2.5) = 3
trunc(-2.7) = -2, fract(2.75) = 0.75
abs(-3.5) = 3.5, signum(-3.5) = -1
sqrt(2) = 1.414214
2^10 = 1024, 2^0.5 = 1.414214
e^1 = 2.718282, ln(10) = 2.302585, log10(1000) = 3
PI = 3.141592653589793, EPSILON = 2.220446049250313e-16
2.9 as i32         -> 2
2.9.round() as i32 -> 3


## 9. `char`, `&str` and `String`: A First Look

Rust keeps *what the text is* separate from *who owns the memory*. Chapter 05 is
dedicated to ownership; chapter 04 is dedicated to text. What follows is the
minimum needed to read the rest of this notebook.

| Type | Meaning | Notes |
|---|---|---|
| `char` | one Unicode **scalar value** | always 4 bytes; `'a'`, `'ß'`, `'😀'` |
| `&str` | a **borrowed view** of UTF-8 bytes | string literals are `&'static str` |
| `String` | an **owned**, growable UTF-8 buffer | heap-allocated; `push_str`, `push` |

Three differences from Python and C++ worth stating outright:

- **There is no indexing.** `text[0]` does not compile, in either form. You cannot
  get at "the third character" in constant time, because in UTF-8 the third
  character is not at a predictable byte offset.
- **A `String` is not a `&str`**, and this is not an annoyance you defeat with a
  cast. It is the ownership distinction showing through: one of them owns the
  bytes, the other merely points at somebody else's.
- **Text is UTF-8**, so `.len()` is **bytes**. `"héllo 😀".len()` is 11 while its
  character count is 7. Everything that iterates a string does so in bytes unless
  you explicitly ask for `.chars()`.

`char` deserves its own warning. It is an *Unicode scalar value*, not a
user-perceived character: `'é'` can be a single `char` or a base `'e'` followed by a
combining accent, depending on how the text was produced. Grapheme clusters are a
higher-level concept that `std` does not provide — reach for a crate when you need
it.

Most functions that take text should take `&str`, because a `&str` accepts both
borrowed and owned text: `&String` coerces to `&str` automatically. Taking
`String` forces every caller to hand over ownership; taking `&str` does not.


In [ ]:
// 9.1 `char` is a Unicode code point, not a byte.

let c: char = 'a';
println!("{c} alphabetic: {}", c.is_alphabetic());
println!("{c} is_uppercase: {} -> {}", c.is_uppercase(), c.to_ascii_uppercase());
println!("'5'.to_digit(10) = {:?}", '5'.to_digit(10));      // Some(5)
println!("'a' as u32      = {}", c as u32);                 // 97
println!("size_of::<char>() = {}", std::mem::size_of::<char>());


a alphabetic: true
a is_uppercase: false -> A
'5'.to_digit(10) = Some(5)
'a' as u32      = 97
size_of::<char>() = 4


In [ ]:
// 9.2 `&str` (borrowed) vs `String` (owned).

let literal: &str = "hello";                  // literals are &'static str
let owned: String = String::from(literal);    // allocates and copies
let also_owned: String = literal.to_string();
println!("{literal} / {owned} / {also_owned}");

// `+` takes an owned String on the left and a &str on the right. The asymmetry
// is deliberate: it reuses the left buffer instead of allocating a third one.
let greeting = String::from("Hello, ") + literal + "!";
println!("{greeting}");

// A String borrows as &str for free — that is the type most APIs want.
let borrowed: &str = &owned;
println!("borrowed = {borrowed}, as_str() = {}", owned.as_str());


Error: The variable `borrowed` contains a reference with a non-static lifetime so
can't be persisted. You can prevent this error by making sure that the
variable goes out of scope - i.e. wrapping the code in {}.

In [ ]:
// 9.3 Length is bytes. Characters need `.chars()`.

let ascii = "hello";
let mixed = "héllo 😀";
println!("{ascii:>8}: bytes = {}, chars = {}",
         ascii.len(), ascii.chars().count());
println!("{mixed:>8}: bytes = {}, chars = {}",
         mixed.len(), mixed.chars().count());
println!("{mixed} is_ascii = {}", mixed.is_ascii());

// Slicing is by BYTE range and panics on a non-char boundary:
//     &mixed[0..2]     // panic: byte index 2 is not a char boundary
println!("&mixed[0..1] = {:?}", &mixed[0..1]);      // "h"
println!("first char   = {:?}", mixed.chars().next());


## 10. Printing and Formatting

Four printing macros:

| Macro | Destination | Newline |
|---|---|---|
| `println!` | stdout | yes |
| `print!` | stdout | no |
| `eprintln!` | stderr | yes |
| `format!` | returns a `String` | — |

The format-string mini-language is worth learning properly, because you will use it
in every notebook from here on:

| Spec | Meaning |
|---|---|
| `{}` | `Display`, the human-readable form |
| `{:?}` / `{:#?}` | `Debug`, compact / pretty |
| `{:<8}` `{:>8}` `{:^8}` | left / right / centre in a field of width 8 |
| `{:0>6}` | right-aligned in width 6, padded with `0` |
| `{:-<10}` | left-aligned in width 10, padded with `-` |
| `{:.3}` | precision: 3 decimals for floats, 3 chars for strings |
| `{:+.2}` | always print the sign |
| `{:b}` `{:o}` `{:x}` `{:X}` | binary / octal / hex lowercase / hex uppercase |
| `{:e}` `{:E}` | scientific notation |
| `{n}` | inline capture: uses the variable `n` in scope |
| `{0} {1}` | explicit positional arguments |
| `{who}` with `who = ...` | named arguments |
| `{{` `}}` | literal braces |

Widths and precisions can themselves be arguments: `{value:>width$.prec$}`.

A few notes that save time later:

- **The format string is checked at compile time.** A misspelled variable or a
  missing argument is a compile error, not a runtime one, and the literal parts are
  borrowed rather than copied.
- **`{}` and `{:?}` are not interchangeable.** `{}` requires the `Display` trait,
  which is deliberately not auto-derived; `{:?}` requires `Debug`, which almost
  always is. When a type "has no Display", that is a design statement: the author
  has not decided what its user-facing rendering should be.
- **`println!` panics if the write fails** — for example when its output pipe has
  been closed, as happens in `prog | head`. `writeln!` into a handled `io::Write`
  returns a `Result` instead, which is the pattern chapter 07 uses.
- **stdout is line-buffered**, so `print!` without a newline may not appear until
  something flushes — and stdout and stderr are separate streams that can interleave
  differently than you wrote them. When order matters, send everything to one
  stream.


In [ ]:
// 10.1 The four macros, and the three ways to name arguments.

println!("println! adds a newline");
print!("print! does not");
print!(" - see?\n");
eprintln!("eprintln! writes to stderr");

let name = "Ada";
let age = 36;
let flags = (true, false);

println!("{name} is {age}");                 // inline capture
println!("{} is {}", name, age);             // positional
println!("{0} is {1}, and {0} again", name, age);   // reuse position 0
println!("{who} is {how_old}", who = name, how_old = age);   // named
println!("Debug: {flags:?}   Display: {}", flags.0);

// A trailing `,` is allowed after the last argument. Note that there is no
// `{x=}` shorthand for printing "name = value" (that is a Python f-string
// feature) -- in Rust you write the label yourself:
let width = 42;
println!("width = {width:?}   vs   {width}   vs   {}", width + 1);


In [ ]:
// 10.2 Width, alignment, fill and precision.

let items = [("coffee", 3.5), ("sandwich", 12.0), ("tea", 2.25)];

println!("{:<10} {:>8}", "ITEM", "PRICE");
println!("{:-<10} {:->8}", "", "");          // fill + align + width
for (item, price) in items {
    println!("{:<10} {:>8.2}", item, price);
}

println!("{:.3}", std::f64::consts::PI);      // precision
println!("{:^11}|", "centered");              // centre alignment
println!("{:0>6}", 42);                       // zero padding
println!("{:+.3}", -1.5);                     // forced sign
println!("{:*^13}", " filled ");              // custom fill + centre

// Precision on a string truncates it.
println!("{:.5}", "truncation");


In [ ]:
// 10.3 Radix, prefixes, scientific notation and computed width.
// (Alignment, fill and padding were §10.2.)

let n = 255u32;
println!("dec {n} | hex {n:x} | HEX {n:X} | oct {n:o} | bin {n:b}");

// `#` inserts the radix prefix; `010` zero-fills to a total width of 10.
println!("{n:#010b}");
println!("{n:#x}");

let big: f64 = 12345.6789;
println!("scientific {big:e} | upper {big:E} | general {big} | rounded {big:.2}");

// A literal brace is written twice.
println!("literal braces: {{ }} around a value {n}");

// Width and precision may be given by name, using `name$`.
let pi = 3.14159265358979;
let width = 10;
let prec = 4;
println!("[{pi:>width$.prec$}]");
println!("{pi:.prec$} with prec = {prec}");

// Width interacts surprisingly with `:?`: it is honoured for numbers and for
// the *elements* of a container, but a bare string ignores it entirely.
println!("[{:>8?}]", 42);        // [      42]
println!("[{:>8}]", "hi");       // [      hi]      <- Display pads
println!("[{:>8?}]", "hi");      // ["hi"]          <- Debug of str does not
println!("[{:>8?}]", [1, 2]);    // [[       1,        2]]

println!("throughput {value:>+12.3}", value = 12345.6789);
println!("({:^26})", "a centred column header");


## 11. `format!`, `dbg!` and Debug vs Display

- `format!` is `println!` without the printing — it **returns a `String`**. Use it
  to build messages, filenames, table rows, error text. It shares the entire
  format grammar with §10, so everything you learned there applies unchanged.
- `dbg!` prints `file:line expr = value` to **stderr** and **returns the value it
  was given**. Because it evaluates to its own argument, it can be dropped into the
  middle of an expression without restructuring anything. It is the fastest way to
  find out what a value actually is — and, unlike a hand-written `println!`, it is
  impossible to mislabel, since the expression is printed verbatim.
- `Display` (`{}`) is the **user-facing** rendering. Only types that implement it
  can use `{}` — every primitive you have met so far does.
- `Debug` (`{:?}`) is the **programmer-facing** rendering, and it can almost always
  be obtained automatically with `#[derive(Debug)]` on your own types (chapter 10).

The distinction is a design decision worth respecting rather than working around:
`Debug` output is allowed to change at any time, so **never parse it and never show
it to an end user**. If a type of yours has a meaningful user-facing form, implement
`Display` by hand and let `Debug` stay the diagnostic view.

`to_string()` and `.to_owned()` on any `Display` type produce a `String`, and
`String` itself can be pushed to, unlike `&str`.

One caution: `dbg!` is a debugging tool, not a logging facility. It writes to
stderr and returns the value, so leaving one in a hot loop is a real performance
cost. Take it out before you call something finished.


In [ ]:
// 11.1 format! builds a String.

let who = "world";
let msg: String = format!("Hello, {who}!");
println!("{msg} ({} bytes)", msg.len());

// Building a small table row by hand is a very common use.
let (label, value) = ("throughput", 12_345.6789);
let row = format!("{label:<12}|{value:>12.3}|");
println!("{row}");
println!("({} columns wide)", row.len());


In [ ]:
// 11.2 dbg! returns its argument, so it fits inside an expression.

let doubled = dbg!(21 * 2);
println!("doubled = {doubled}");

let pairs = [(1, "one"), (2, "two")];
let first = dbg!(pairs[0]);          // prints the file/line and the value
println!("first = {first:?}");

// Because it sits in an expression, it also works mid-pipeline:
let total = [1, 2, 3].iter().map(|x| dbg!(x * 5)).sum::<i32>();
println!("total = {total}");


In [ ]:
// 11.3 Debug vs Display.

// `{:?}` works on essentially everything, including nested structures.
let mixed = (1u8, 2.5f64, 'x', "end", [1, 2, 3]);
println!("{mixed:?}");       // Debug, compact
println!("{mixed:#?}");      // Debug, pretty (multi-line)

// `{}` needs Display. Primitives, &str, String and tuples of such types have it;
// your own structs will not until you implement it (chapter 10).
let nested = vec!["a", "b", "c"];      // a first peek at Vec — chapter 06
println!("Display of joint: {}", nested.join(", "));

// Trying `{}` on something without Display gives:
//     error[E0277]: `(i32, &str)` doesn't implement `std::fmt::Display`
// Fix: use `{:?}`, or implement Display yourself.


## 12. Tuples and Arrays

Both are **fixed-size compound types**, and both are stored inline rather than on
the heap. Neither grows.

| | Tuple `(A, B, ...)` | Array `[T; N]` |
|---|---|---|
| Elements | may differ in type | all the same type |
| Length | part of the type | part of the type |
| Access | `.0`, `.1`, or destructuring | `arr[i]`, or destructuring |
| Out-of-range access | impossible statically | panics (or use `.get()`) |

The word *fixed* is the important one: because the length is part of the type, the
compiler knows it, and an array of three elements is not the same type as an array
of four. That is what lets bounds be checked and makes slices describable.

Notes:

- Tuples are the idiomatic way to return several values from a function. There is no
  "multiple return" syntax; there is a tuple, and destructuring on the way out.
- A one-element tuple needs a trailing comma: `(42,)` — without it you just have
  parenthesised `42`. (And note that `(42)` has type `i32`, whereas `(42,)` has type
  `(i32,)`.)
- `.0 .1` needs a space when you chain field access on a nested tuple, because
  `.0.1` is parsed as a single float literal.
- `arr[i]` is bounds-checked and panics on failure; `.get(i)` returns `Option` and
  is the honest version when the index is not known to be valid. When the index *is*
  a constant that the compiler can see, it usually removes the check entirely.
- Arrays of `Copy` elements are themselves `Copy`: assigning one **duplicates** it,
  whereas a `Vec` (chapter 06) would move. Arrays of non-`Copy` elements are not
  `Copy` and behave differently under assignment — a distinction that only becomes
  visible once ownership is on the table (chapter 05).
- Arrays are the natural type for fixed data (a 3×3 matrix, a lookup table) but the
  wrong one for anything whose size you learn at runtime — that is `Vec`'s job.
- Slices (`&arr[a..b]`) are borrowed views and are the standard way to pass "some
  contiguous data" around, precisely because they carry their length with them.


In [ ]:
// 12.1 Tuples.

// Heterogeneous, fixed arity.
let point: (i32, i32) = (3, 4);
let mixed = (1u8, "text", 2.5, true);
println!("mixed = {mixed:?}");

// Field access by index.
println!("x = {}, y = {}", point.0, point.1);

// Destructuring binds every part in one statement.
let (x, y) = point;
println!("x + y = {}", x + y);

// Nested tuples (mind the space in `.0 .1` — `.0.1` would parse as a float).
let nested = ((1, 2), (3, 4));
println!("nested.0.1 = {}", nested.0 .1);

// The unit tuple contains nothing; it is the type of "no value".
let nothing: () = ();
println!("unit = {nothing:?}, size = {} byte", std::mem::size_of::<()>());

// A one-element tuple needs the trailing comma.
let one = (42,);
println!("{one:?}");


In [ ]:
// 12.2 Arrays.

// Length is part of the type.
let mut arr: [i32; 4] = [10, 20, 30, 40];
println!("arr = {arr:?}, len = {}", arr.len());

// Indexing panics when out of range.
println!("arr[2] = {}", arr[2]);
// println!("{}", arr[9]);              // panic: index out of bounds

// `.get` returns an Option instead.
println!("arr.get(2) = {:?}, arr.get(9) = {:?}", arr.get(2), arr.get(9));

// Arrays of Copy elements are Copy: assignment duplicates.
let copy = arr;
arr[0] = 99;
println!("arr  = {arr:?}");
println!("copy = {copy:?}   <- unaffected");

// Slicing borrows a contiguous view; the slice is the idiomatic
// "some data" parameter type.
let middle = &arr[1..3];
println!("&arr[1..3] = {middle:?} (len {})", middle.len());

// Repeat syntax.
let zeros = [0u8; 5];
let grid = [[0i32; 3]; 2];            // 2 rows of 3 columns
println!("zeros = {zeros:?}");
println!("grid  = {grid:?}");


## 13. Constants, Statics and Type Aliases

| Item | Evaluated | Stored | Use for |
|---|---|---|---|
| `let` | at runtime | on the stack (usually) | ordinary values |
| `const` | at **compile time** | inlined at each use site | real constants, array lengths |
| `static` | at compile time | one fixed memory location | global state, `&'static` data |
| `type` | — | nothing | a nickname for an existing type |

- `const` and `static` require an **explicit type** — no inference. This is not
  pedantry: a constant is part of an interface, so its type should be a decision
  somebody made.
- Naming convention is `SCREAMING_SNAKE_CASE` for both.
- `const` values may be used where the compiler needs a literal, e.g. `[0u8; N]`.
- `type` aliases are transparent: `type Celsius = f64;` means `Celsius` **is** `f64`,
  with all its methods, and you can pass one where the other is expected. (To get a
  genuinely distinct type you need a newtype struct — chapter 10.)
- `const fn` can be evaluated at compile time, which is how non-trivial `const`
  values are built. A `const fn` is also an ordinary function, callable at runtime.

The `const` / `static` split is subtle and worth understanding now:

- A `const` **has no address**. Each use site gets its own copy of the value, inlined
  as though you had typed the literal there. That makes it free to use and impossible
  to take a stable reference to.
- A `static` **has exactly one address** for the whole program, so `&APP_NAME` is a
  real `&'static str` you can hand to anything, including a thread. The price is that
  the compiler requires `Sync` for a shared `static`, and mutating one is `unsafe` —
  which is precisely why Rust code tends to avoid global mutable state.
- Both are zero-cost compared to a local `let`, and both are *better* than a local
  when the value is genuinely a constant: the name gets reused, and the value is not
  reconstructed on every call.


In [ ]:
// 13.1 const, static, type.

const MAX_RETRIES: u32 = 5;
const GREETING: &str = "hi";
static APP_NAME: &str = "training-rust";

println!("{GREETING} from {APP_NAME}: retry up to {MAX_RETRIES} times");

// A `const` can be used where a compile-time value is required.
const ROWS: usize = 3;
let table = [0u8; ROWS];
println!("table has {} rows", table.len());

// `type` is a transparent nickname, not a new type.
type Celsius = f64;
type Pair = (f64, f64);
let boiling: Celsius = 100.0;
let p: Pair = (1.0, 2.0);
// Transparent means this is legal:
let as_plain_f64: f64 = boiling;
println!("{boiling}°C == {as_plain_f64}°C, {p:?}");

// `const fn` is evaluated at compile time, which is a preview of chapter 03.
const fn doubled(x: i32) -> i32 { x * 2 }
const SIX: i32 = doubled(3);
println!("doubled(3) computed at compile time = {SIX}");


## 14. Expressions, Statements and Blocks

This is the single most important structural idea in Rust, and it explains a lot of
later syntax at once.

- **Rust is expression-oriented.** Almost everything produces a value, including
  blocks, `if`, `match`, `loop`, and function bodies.
- A **block** `{ ... }` is an expression. Its value is its **last expression**,
  provided that expression has **no trailing semicolon**.
- Adding a `;` turns the last expression into a **statement**, so the block then
  evaluates to `()`, the unit value.
- Consequently a function body is just a block expression — there is no need for
  `return`, except to exit early.
- Statements (`let`, item definitions) are not expressions and cannot be used where
  a value is expected. `let x = (let y = 1);` is a syntax error, not a nil.

What this buys you is uniformity: initialising a binding, choosing between two
values, and computing a result inside a nested scope all look the same, and all of
them fit into a single `let`:

``` rust
// Rust: the `if` you already know IS the conditional operator.
let status = if area > 10 { "big" } else { "small" };
```

```python
# Python, for contrast: a dedicated expression form.
status = "big" if area > 10 else "small"
```

```cpp
// C++: a separate corner of the grammar, one of the few ternary operators left.
auto status = area > 10 ? "big" : "small";
```

The trap is the semicolon, and it is worth stating as a rule: **a stray `;` at the
end of a function body is the classic beginner bug**, because it changes the body's
type from `T` to `()`. The compiler's complaint — `expected i32, found ()` — points
at the closing brace rather than at the semicolon, which is what makes it confusing
the first time.

Blocks also introduce **scopes**: a binding made inside a block disappears at its
closing brace, and §3's shadowing is what makes that feel natural rather than
restrictive.


In [ ]:
// 14.1 A block's value is its final expression.

let area = {
    let w = 3;
    let h = 4;
    w * h                    // no semicolon -> this is the block's value
};
println!("area = {area}");

// With a trailing semicolon the expression becomes a statement, so the value
// is discarded and the block yields ().
//
// Writing plain `w * h;` here would compile, but the compiler would warn:
//     warning: unused arithmetic operation that must be used
// `let _ = ...` is the idiomatic way to say "yes, discarded on purpose".
let nothing = {
    let w = 3;
    let h = 4;
    let _ = w * h;           // value intentionally thrown away
};
println!("nothing = {nothing:?}");

// Which is exactly the difference between these two function bodies:
fn with_value() -> i32 { 7 }        // last expression, no semicolon

// fn without_value() -> i32 { 7; } // error[E0308]: expected i32, found ()
//                                  // the `;` threw the 7 away

println!("with_value() = {}", with_value());

// A block makes a new scope; inner bindings disappear at the closing brace.
let outer = 1;
{
    let outer = "shadowed inside";
    println!("{outer}");
}
println!("{outer}");


In [ ]:
// 14.2 `if` is an expression too (chapter 02 takes this further).

let area = 12;
let status = if area > 10 { "big" } else { "small" };
println!("status = {status}");

// Both branches must have the same type, because the `if` has one type:
//     let bad = if area > 10 { 1 } else { "small" };
//     error[E0308]: `if` and `else` have incompatible types

// Statements are not expressions:
//     let x = (let y = 1);      // error: expected expression, found `let` statement

// A stray semicolon at the end of a function is the classic beginner bug.
// This body does NOT compile, so it is shown commented out:
//
//     fn perimeter(w: f64, h: f64) -> f64 {
//         2.0 * (w + h);              // error[E0308]: expected `f64`, found `()`
//     }
//
// The fix is to drop the `;`:
fn perimeter(w: f64, h: f64) -> f64 {
    2.0 * (w + h)
}
println!("perimeter(3.0, 4.0) = {}", perimeter(3.0, 4.0));


## 15. Reading the Compiler

Rust's error messages are unusually good, and reading them properly is a real
skill — one that pays off every day. The anatomy:

```
error[E0384]: cannot assign twice to immutable variable `x`
  --> src/main.rs:3:5
   |
 2 |     let x = 5;
   |         - first assignment to `x`
 3 |     x = 6;
   |     ^^^^^ cannot assign twice to immutable variable
   |
help: consider making this binding mutable
   |
 2 |     let mut x = 5;
   |         +++
```

Read it in this order:

1. **The last line of the message.** It is written as a sentence about your code,
   and it usually names the problem exactly: *cannot assign twice to immutable
   variable `x`*.
2. **The `help:` and `note:` blocks.** These frequently contain the fix verbatim,
   including the `+++` markers showing exactly what to insert. When a `help:` exists,
   read it before you invent a solution.
3. **The error code** (`E0384`). It is a documentation index: `rustc --explain E0384`
   prints the full explanation, and the same page is online. Every code is
   explainable, which means you are never stuck on a message you cannot decode.
4. **The earlier markers, not only the caret.** The line marked with a plain `-` is
   where the compiler thinks the *cause* lives. For an immutable-assignment error
   that is the `let`; for a type error it is the annotation you supplied. The fix is
   usually up there, not on the line that got flagged.

And here is a type mismatch, the most common error you will meet:

```
error[E0308]: mismatched types
 --> src/main.rs:3:18
  |
3 |     let b: i64 = 1i32;
  |            ---   ^^^^ expected `i64`, found `i32`
  |            |
  |            expected due to this
```

Rust states *expected* before *found*, which is the opposite of what most people
expect: the annotation you wrote is the "expected" side. Read those two words
carefully and you can diagnose most of the E0308 family without thinking.

Some housekeeping that will save you time:

- **Errors cascade.** One mistake — a missing `;`, an undefined name — often
  produces a burst of unrelated-looking complaints. Fix the **first** error, then
  recompile; the rest are frequently fiction.
- **`cargo check` is the fast loop.** It type-checks the crate without generating
  code, so it is several times quicker than `cargo build`. Use it while iterating,
  and build only when you actually want the binary.
- **Warnings are not noise.** `unused variable`, `unused_mut` and `unused_imports`
  are all pointing at real sloppiness. Prefix a binding with `_` when it is
  intentionally unused, rather than leaving the warning in place.
- **The borrow checker has its own family of codes** — `E0382` (use of moved value),
  `E0502` (conflicting borrows), `E0499` — and they arrive with a "cannot borrow
  here" label and a note explaining what still holds the value. Chapter 05 covers
  the semantics; the messages are worth meeting early, because they are the ones
  that most often look impossible the first time.
- Toolchain: `cargo check` for a fast type-check with no code generation,
  `cargo build` / `cargo run [--release]`, `cargo test`, `cargo clippy` for lints,
  `cargo fmt` for formatting.

If you use this notebook against a plain `rustc` invocation rather than cargo, every
one of those flags still applies — `rustc --explain`, `rustc -A warnings` to silence
a category you are deliberately ignoring, `cargo check`'s equivalent being
`rustc --emit=metadata`.


In [ ]:
// 15.1 The compiler as a teacher — common beginner diagnostics.

// 1) Unused binding
//      warning: unused variable: `total`
//    Fix: use it, or prefix it with an underscore.
let _total = 10;

// 2) Immutable assignment
//      error[E0384]: cannot assign twice to immutable variable
//    Fix: declare with `mut`.
let mut fixed = 1;
fixed += 1;
println!("fixed = {fixed}");

// 3) Type mismatch
//      error[E0308]: mismatched types
//    Fix: convert explicitly.
let a: i32 = 5;
let b: i64 = i64::from(a);
println!("a = {a} ({} bytes), b = {b} ({} bytes)",
         std::mem::size_of_val(&a), std::mem::size_of_val(&b));

// 4) Literal out of range for its type
//      error: literal out of range for `u8`
//    Fix: widen the type.
let wide_enough: u16 = 300;
println!("wide_enough = {wide_enough}");

// 5) Tuple arity mismatch
//      error[E0308]: expected a tuple with 2 elements, found one with 3 elements
let pair: (i32, i32) = (1, 2);
println!("pair = {pair:?}");

// 6) Missing semicolon in the middle of a function body
//      error: expected `;`, found ...
//    Fix: add it — or, if it was the last expression, add it and then supply a
//         value, as §14 showed.

println!("\nThe toolkit: cargo check, build, run, test, clippy, fmt.");


## 16. Cheat Sheet

**Program shape**

``` rust
fn main() {              // entry point
    let x = 5;           // binding, immutable
    println!("{x}");     // macro, braces interpolate
}
```

**Bindings**

| Syntax | Effect |
|---|---|
| `let x = 1;` | immutable binding |
| `let mut x = 1;` | mutable binding |
| `let x = 1; let x = "a";` | shadowing, type may change |
| `let x: u8 = 1;` | explicit type |
| `let _x = 1;` | explicitly unused |

**Scalars** — `i8 i16 i32 i64 i128 isize`, `u8 u16 u32 u64 u128 usize`,
`f32 f64`, `bool`, `char`. Defaults: `i32` and `f64`. Indices and lengths are `usize`.

**Literals** — `1_000_000`, `0xFF`, `0o755`, `0b1010`, `b'A'`, `10u8`, `1.5e-3`.

**Operators** — `- !` · `* / %` · `+ -` · `<< >>` · `& ^ |` · comparisons · `&& ||`.
Bitwise binds **tighter** than comparison. `&&`/`||` short-circuit; `&`/`|` do not.

**Conversions** — `x as u8` (blunt, saturating for floats), `u64::from(x)` /
`x.into()` (checked where possible), `u8::try_from(x)` (returns `Result`).
Arithmetic never converts implicitly.

**Overflow** — `checked_*` → `Option`, `overflowing_*` → `(v, bool)`,
`saturating_*` → clamped, `wrapping_*` → wrapped. Plain `+ - *` panics in debug and
wraps in release.

**Absence and failure** — `Option<T>` (`Some`/`None`), `Result<T, E>` (`Ok`/`Err`).
`unwrap()` panics; `unwrap_or(default)` does not.

**Text** — `char` (4 bytes, one code point), `&str` (borrowed, literals), `String`
(owned, growable). `.len()` is bytes; `.chars().count()` is characters.

**Formatting** — `{} {} {:?} {:#?} {:<8} {:>8} {:^8} {:0>6} {:-<10} {:.3} {:+.2}
{:b} {:o} {:x} {:X} {:e} {name} {0} {{ }}`.

**Macros** — `println!` / `print!` / `eprintln!` / `format!` (returns `String`) /
`dbg!` (stderr, returns its argument).

**Compounds** — tuples `(A, B)` with `.0`/destructuring; arrays `[T; N]` with
`arr[i]` (panics) or `.get(i)` → `Option`; slices `&arr[a..b]`.

**Items** — `const NAME: T = ...;`, `static NAME: T = ...;`, `type Nickname = T;`,
`const fn f() -> T { ... }`.

**Expressions** — a block yields its last expression; a `;` discards it and yields `()`.

### Coming from Python or C++

| You would write | In Rust | Why |
|---|---|---|
| `x = 1` | `let mut x = 1;` | immutability is the default |
| `None` / `nullptr` | `Option<T>` | no null; absence is in the type |
| `try/except` | `Result<T, E>` | errors are values, not control flow |
| `int` / `long` | `i32` / `i64` | width is explicit, and does not promote |
| `1 + 1.0` | `1i32 as f64 + 1.0` | no implicit numeric conversion |
| `type(x)` | `type_name_of_val(&x)` (debug) | no runtime type objects |
| `x is int` | generics or a trait | dispatch is resolved at compile time |
| `f"hi {name}"` | `format!("hi {name}")` | a macro checked at compile time |
| `f"{x=}"` | `dbg!(x)` | and it prints the source expression |
| `len(s)`, `s[0]` | `s.len()` (bytes); no indexing | UTF-8 makes indexing meaningless |
| `d = {}` | `HashMap::new()` | no literals; see chapter 06 |
| `s[:3]` | `&s[..3]` | byte ranges, bounds-checked |
| `a if c else b` | `if c { a } else { b }` | `if` is an expression |
| `sprintf` / `<<` | `format!` | one formatting grammar for everything |
| `printf` / `cout` | `println!` | a macro, compile-time checked |
| `for (k,v) in d.items()` | `for (k, v) in &map` | iteration by reference is explicit |
| `#include` | `use` | and crates are declared in `Cargo.toml` |
| `[]` on a map | `.get(&k)` → `Option<&V>` | a missing key is not an exception |
| `true`/`false` as 1/0 | `u8::from(b)` | no truthiness: `if 1` will not compile |
| `var x = 1` (C++) | `let x = 1;` | and `let mut` when it must change |
| `auto` | `let` (inferred) | inference is the norm, not an option |
| `delete`/`free` | nothing | ownership does it (chapter 05) |


### What Comes Next

- **Practice:** `01_syntax-printing-and-types_p.ipynb` — exercises mirroring the
  sections above, with the answer cells left for you to fill in. The section and
  exercise numbering line up with this notebook, so when you get stuck you can jump
  straight back to the matching section.
- **Chapter 02:** logic blocks — `if`/`else`, `loop`, `while`, `for`, ranges, `match`.
  The `if`-as-expression idea from §14 gets a full treatment there, and `Option` and
  `Result` from §7.4 finally get an honest way to be taken apart.
- **Chapter 03:** function declaration, parameters, return values, and multiple
  return values via tuples.
- **Chapter 04:** text properly — `String` methods, `&str` borrowing, slicing, and
  the character/byte distinction introduced in §9.
- **Chapter 05:** ownership and borrowing, which is what makes §9's `&str` versus
  `String` split make sense, and where `Option` is used for real.

Three things worth carrying forward from this chapter:

1. **Immutability is the default**, and that is a design feature rather than a
   hurdle to clear with `mut`.
2. **Types are explicit at every boundary.** No promotions, no truthiness, no
   surprises — the errors you get for mixing types are the compiler refusing to
   guess what you meant.
3. **Diagnostics are part of the language.** When something fails to compile, read
   the message before touching the code; it is usually a better explanation than the
   one you would invent.

If something in this notebook did not compile when you ran it, that is a bug in the
notebook, not in your understanding — worth reporting.
